In [ ]:
# 1. サンプルデータセットをダウンロード
%cd /content
!git clone https://github.com/wal-afk/drive_sim
%cd drive_sim
!git pull
!git restore .
!git clean -fd
%cd /content/drive_sim

!pip install -U plotly==6.9

In [ ]:
import yaml

from sim.drive_simulator import CarSim
from sim.vehicle import VehicleProp
from sim.mission_base import MissionBase
from sim.goal import GoalLine, GoalCircle
from sim.drawer import SimDrawer, MissionDrawer
from sim.worlds.type_b_world import type_b_circuit
from sim.sign import Sign

with open("config/type-b.yaml", "r") as f:
    vehicle_config = yaml.safe_load(f)

prop = VehicleProp(**vehicle_config)

# プログラムの書き方講座4

## スムーズな動作にしよう

- チュートリアル3では、標識をチェックしながら車を前進させるために、下記のように繰り返していた
  - 標識をチェックする
  - 標識がぶつかりそうにないなら10cm進む
  - 標識をチェックする
  - 標識がぶつかりそうにないなら10cm進む
  - ・・・(繰り返し)
- この方法には問題がある。どんな問題が思いつく？

- スムーズに動かすには下記の方が望ましい
    ```
    標識を見つけよ
    - 標識にぶつかりそうなら止まれ
    - 標識にぶつかりそうにないなら決まった前進速度を維持
    最初に戻って繰り返せ
    ```
- こうしておくと、コンピューターの能力を最大限活用した速さで標識に気付いて止まることができる

## プログラムを改善すると・・・

- 下記の「ほぼ正面に虫を見つけたら、虫の方向に前進する」プログラムをスムーズに動作するように書くと下記になる。何が違うか比べてみよう。
- 改善前　：　ほぼ正面に虫がいる限り0.2[m]づつ前進し続ける
  ```
  while True:
    pos = search()
    if pos is None:
      move(v=0)
    else:
      if pos.theta <= 5 and pos.theta >= -5:
        move(v=0.2, t=1.0)
        wait()
      else:
        move(v=0)
  ```
- 改善後　：　ほぼ正面に虫がいる限り、0.2[m/秒]で前進し続ける
  ```
  while True:
    pos = search()
    if pos is None:
      move(v=0)
    else:
      if pos.theta <= 5 and pos.theta >= -5:
        move(v=0.2)
      else:
        move(v=0)
  ```

- ポイント
    - 0.2[m]進むには時間がかかる（1秒程度）のに対して、速度を0.2[m/秒]に設定するのは一瞬で終わる
    - wait()を使わないことで、すぐに次の繰り返しが始まる
        - コンピュータの処理能力によるが、例えば10分の1秒～100分の1秒の速さで認識に応じた車のコントールもできる


# チュートリアル4

下記の命令を組み合わせてプログラムを書き、ロボットを直進させながら標識(標識名はsign1)が車にぶつかりそう（左右0.2[m]以内）に見えた瞬間に停止しよう。※問題はチュートリアル3と同じ

## 取り組み方
1. 使える命令を理解する
    - チュートリアル3と同じだがwaitは使用不可
2. 下のセルを実行して、ロボットの限界速度や、ロボットが存在する初期位置やチェックポイント（goal）を把握する
3. ２つ下のセル内にプログラムを書き実行して結果を見る

|使える命令|意味|指定できる値|使い方|
|--|--|--|--|
|move|一定速度で前に進む|v=速度[m/s]|move(v=0.2)|
|search|標識を見つける（複数見つかった場合は、最も近いもの）|-|pos = Search()|

- pos = search() が返す値には下記が含まれる
  - pos.x: 見つけた標識の前方位置[m]　※前方が正
  - pox.y: 見つけた標識の左右位置[m]　※左側が正、右側は負
  - pos.r: 見つけた標識への距離[m]
  - pos.theta: 見つけた標識の角度[度]　※左側が正、右側は負
  - pos.name: 見つけた標識の標識名

## 注意点
- スタート時の位置はランダムに前後最大30cmほどずれる（左右ずれはない）

In [ ]:
class Tutorial4(MissionBase):
    def __init__(self):
        super().__init__(type_b_circuit, t_max=20)
        self.goals = [
            GoalCircle((2.2, 0.0), 0.2, should_stop=True),
        ]
        self.initial_xy = (0.0, 0.0)
        self.random_d_xy = (0.3, 0.0)
        self.set_signs(
            [
                Sign(x=1.9, y=-0.4, name="sign1"),
                Sign(x=2.7, y=0.4, name="sign1"),
                Sign(x=3.5, y=-0.1, name="sign1"),
            ]
        )


MissionDrawer(Tutorial4()).show()
print("最大速度", prop.max_velocity, "m/s")
print("最大回転速度", prop.max_rotate_deg, "度/s")

In [ ]:
class Tutorial4(MissionBase):
    def __init__(self):
        super().__init__(type_b_circuit, t_max=20)
        self.goals = [
            GoalCircle((2.2, 0.0), 0.2, should_stop=True),
        ]
        self.initial_xy = (0.0, 0.0)
        self.random_d_xy = (0.3, 0.0)
        self.set_signs(
            [
                Sign(x=1.9, y=-0.4, name="sign1"),
                Sign(x=2.7, y=0.4, name="sign1"),
                Sign(x=3.5, y=-0.1, name="sign1"),
            ]
        )

    @staticmethod
    def command_func(*, move, search, **kwargs):
        while True:
            pos = search(name="sign1")
            if pos is None:
                move(v=0.2)
            else:
                # ####### ここから下に「標識が左右0.2[m]以内なら停止し、そうでないなら一定速度で前進する」プログラムを書こう
                move(v=0.0)
                # ####### ここより上にプログラムを書こう
        # ####### プログラムを書いた後にセルを実行し結果を確認しよう


sim = CarSim(prop, Tutorial4())
sim.run()
SimDrawer(sim).show()